# Preamble

In [3]:
import os
import splitfolders
import cv2 as cv
import numpy as np
import random
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
import pillow_heif  # Ensure HEIC support

%matplotlib inline

# Image Manipulation

In [ ]:
#img = cv.imread('/content/drive/MyDrive/NAIC/IMAGES/testOne/testOne/Kuih Ketayap/Copy of ketayap 1.jpg')
#img = cv.resize(img, (224, 224))
#plt.imshow(img[:,:,::-1])

1. Translation

In [ ]:
def translate(img, x, y):
    transMat = np.float32([[1,0,x],[0,1,y]])
    dimensions = (img.shape[1], img.shape[0])
    return cv.warpAffine(img, transMat, dimensions)

#translated = translate(img, np.random.randint(-75,75),np.random.randint(-75,75))
#plt.imshow(translated[:,:,::-1])

2. Random Rotation

In [ ]:
def rotation(img, angle):
    angle = int(random.uniform(-angle, angle))
    h, w = img.shape[:2]
    M = cv.getRotationMatrix2D((int(w/2), int(h/2)), angle, 1)
    img = cv.warpAffine(img, M, (w, h))
    return img

#Rotated = rotation(img, 45)
#cv.imshow('Rotated image', Rotated)

3. Flip

In [ ]:
def flip(img, flip_code):
    return cv.flip(img, flip_code)

# Horizontal flipping: 0
# Vertical flipping: 1

# 3(a). Horizontal Flipping
#flip = cv.flip(img, 1)
#plt.imshow(flip)

# 3(b). Vertical Flip
#vert_flip = cv.flip(img, 0)
#cv.imshow("Vertical flip", vert_flip)

4. Random Crop

In [ ]:
def get_random_crop(img):

    # get maximum starting x and y coordinates such that the crop image can
    # have the desired dimensions without getting out of bounds
    max_x = round(img.shape[1]*.4)
    max_y = round(img.shape[0]*.4)

    # get start x and y coordinates
    x = np.random.randint(0, max_x)
    y = np.random.randint(0, max_y)

    # get cropped image
    crop = img[y: y + round(img.shape[0]*.6), x: x + round(img.shape[1]*.6),:]

    #
    # creating a zero-padding cropped image to prevent the use of interpolation
    # resizing
    #
    black_array = np.zeros((img.shape[0],img.shape[1],img.shape[2]),dtype=np.int64)

    y_low_bound = round((img.shape[0]-crop.shape[0])/2)
    y_high_bound = y_low_bound + crop.shape[0]
    x_low_bound = round((img.shape[1]-crop.shape[1])/2)
    x_high_bound = x_low_bound + crop.shape[1]

    black_array[y_low_bound:y_high_bound,x_low_bound:x_high_bound,:] = crop

    return black_array

#cropped = get_random_crop(img)
#plt.imshow(cropped[:,:,::-1])

5. Gaussian Blur

In [ ]:
def blur(img):
    blur = cv.GaussianBlur(img, (9,9), cv.BORDER_DEFAULT)
    return blur

#cv.imshow("Blur", blur)

6. Gaussian Noise

In [ ]:
def add_gaussian_noise(img, mean=0, std=25):
    noise = np.random.normal(mean, std, img.shape).astype(np.uint8)
    noisy_image = cv.add(img, noise)
    return noisy_image

#noisy_image = add_gaussian_noise(img, mean=0, std=10)
#cv.imshow("Noisy", noisy_image)

7. Salt and Pepper Noise

In [ ]:
def add_salt_and_pepper_noise(img, noise_ratio=0.02):
    noisy_image = img.copy()
    h, w, c = noisy_image.shape
    noisy_pixels = int(h * w * noise_ratio)

    for _ in range(noisy_pixels):
        row, col = np.random.randint(0, h), np.random.randint(0, w)
        if np.random.rand() < 0.5:
            noisy_image[row, col] = [0, 0, 0]
        else:
            noisy_image[row, col] = [255, 255, 255]

    return noisy_image

#salt_and_pepper_image = add_salt_and_pepper_noise(img, noise_ratio=0.2)
#cv.imshow("Salt and pepper", salt_and_pepper_image)

8. Color Space Conversion

In [ ]:
def cvtColor(img, color_code):
    return cv.cvtColor(img, color_code)

# to HSV: cv.COLOR_BGR2HSV
# to LAB: cv.COLOR_BGR2LAB
# to RGB: cv.COLOR_BGR2RGB
#
# (a)
#
#hsv = cv.cvtColor(img, cv.COLOR_BGR2HSV)
#cv.imshow("HSV", hsv)
#
# (b)
#lab = cv.cvtColor(img, cv.COLOR_BGR2LAB)
#cv.imshow("LAB", lab)
#
# (c)
#rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
#cv.imshow("RGB", rgb)

9. Brightness

In [ ]:
def incBrightness(img):
    # Adjust the brightness and contrast
    # Adjusts the brightness by adding 10 to each pixel value
    brightness = 10
    # Adjusts the contrast by scaling the pixel values by 2.3
    contrast = 2.3
    image2 = cv.addWeighted(img, contrast, np.zeros(img.shape, img.dtype), 0, brightness)
    return image2

#Save the image
# cv.imwrite('modified_image.jpg', image2)
#Plot the contrast image
#plt.subplot(1, 2, 2) # This figure has 1 row, 2 columns, and this plot is the second plot
#plt.title("Brightness & contrast")
#plt.imshow(image2)
#plt.show()

10. Sharpening

In [ ]:
def sharpen(img):
    # Create the sharpening kernel
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])

    # Sharpen the image
    sharpened_image = cv.filter2D(img, -1, kernel)
    return sharpened_image

#Save the image
#cv.imwrite('sharpened_image.jpg', sharpened_image)

#Plot the sharpened image
#plt.subplot(1, 2, 2)
#plt.title("Sharpening")
#plt.imshow(sharpened_image)
#plt.show()

11. Color Enhancement

In [ ]:
def colorEnhance(img):

    # Convert the image from BGR to HSV color space
    image = cv.cvtColor(img, cv.COLOR_BGR2HSV)

    # Adjust the hue, saturation, and value of the image
    # Adjusts the hue by multiplying it by 0.7
    image[:, :, 0] = image[:, :, 0] * 0.7
    # Adjusts the saturation by multiplying it by 1.5
    image[:, :, 1] = image[:, :, 1] * 1.5
    # Adjusts the value by multiplying it by 0.5
    image[:, :, 2] = image[:, :, 2] * 0.5

    # Convert the image back to BGR color space
    image2 = cv.cvtColor(image, cv.COLOR_HSV2BGR)

    return image2

#Save the image

#cv.imwrite('enhanced coloured.jpg', image2)

#Plot the enhanced image
#plt.subplot(1, 2, 2)
#plt.title("enhanced coloured")
#plt.imshow(image2)
#plt.show()

**Main Function**

In [ ]:
def augment_image(target_image_path):

    # check if image is HEIF, if it is, convert to a cv-compatible format
    base, ext = os.path.splitext(target_image_path)
    if ext.lower() == '.heic':
        heif_file = pillow_heif.read_heif(target_image_path)
        pil_image = Image.frombytes(
            heif_file.mode,
            heif_file.size,
            heif_file.data,
            "raw",
            heif_file.mode,
            heif_file.stride,
        )
        img = cv.cvtColor(np.array(pil_image), cv.COLOR_RGB2BGR)
    else:
        img = cv.imread(target_image_path)

    img = cv.resize(img, (224, 224))
    translated = translate(img, np.random.randint(-75,75),np.random.randint(-75,75))
    rotated = rotation(img,np.random.randint(-90,90))
    flipped = flip(img,np.random.randint(0,2))
    crop = get_random_crop(img)
    blurred = blur(img)
    g_noise = add_gaussian_noise(img,mean=0,std=25)
    s_n_p_noise = add_salt_and_pepper_noise(img,noise_ratio=0.02)
    bgr = cvtColor(img,cv.COLOR_BGR2RGB)
    hsv = cvtColor(img,cv.COLOR_BGR2HSV)
    lab = cvtColor(img,cv.COLOR_BGR2LAB)
    incBright = incBrightness(img)
    sharp = sharpen(img)
    enhanced = colorEnhance(img)

    return[translated,rotated,flipped,crop,blurred,g_noise,s_n_p_noise,bgr,hsv,lab,incBright,sharp,enhanced]


# Image Augmentation

In [ ]:
# imageAugment function takes in a folder of images categorised into classes
# from input_path and outputs them into a new folder after performing data
# augmentation. The function then splits the folder into three different
# folders: train, test, and val according to the ratio in split_ratio.

def imageAugment(input_folder_path,split_ratio):
  #
  # ----------input----------
  #
  input_class_folders = os.listdir(input_folder_path)

  #
  # ----------output----------
  #
  # output folder parent directory
  output_parent = os.path.dirname(input_folder_path)
  # output folder name
  output_folder_name = f'{os.path.basename(input_folder_path)}_Output'
  # output folder path
  output_folder_path = os.path.join(output_parent,output_folder_name)
  # create output folder
  os.makedirs(output_folder_path, exist_ok=True)

  class_no = 0

  for image_class in input_class_folders:

    os.makedirs(os.path.join(output_folder_path, image_class), exist_ok=True)

    images_in_class = os.listdir(os.path.join(input_folder_path,image_class))

    image_no = 0

    for image in images_in_class:

      target_image_path = os.path.join(input_folder_path , image_class , image)
      augmented_images = augment_image(target_image_path)

      augmented_image_no = 0

      for augmented_image in augmented_images:

        augmented_image_name = f'{class_no}_{image_no}_{augmented_image_no}.jpg'

        augmented_image_path = os.path.join(output_folder_path,image_class,augmented_image_name)

        cv.imwrite(augmented_image_path, augmented_image)

        augmented_image_no += 1

      image_no += 1

    class_no += 1

  #
  # ----------split in to 3 folders----------
  #
  split_input_path = output_folder_path

  split_output_folder_name = f'{os.path.basename(input_folder_path)}_Split_Output'
  split_output_path = os.path.join(output_parent,split_output_folder_name)
  os.makedirs(split_output_path, exist_ok=True)

  # split folders and save into split_output_path
  splitfolders.ratio(
    split_input_path,
    output=split_output_path,
    seed=1427,
    ratio=split_ratio,
    group_prefix=None
  )

In [ ]:
# Input folder: one subfolder per class, each holding that class's images.
# (This is the original Colab notebook; src/augment.py is the cleaned CLI version.)
input_path = 'data/raw'

In [ ]:
imageAugment(input_path,(0.8,0.1,0.1))

Copying files: 650 files [00:09, 66.72 files/s]


# Miscellaneous

In [4]:
#
# ---------to-do: error catchers---------
#

# 1.  Check if path exists

directory='./testTwo'

img_path = directory
print("Path exists:", os.path.exists(img_path))  # Check path

if os.path.exists(img_path):
    img = cv.imread(img_path)
    print("Image shape:", img.shape if img is not None else "Image not loaded")
else:
    print("Adjust the file path!")


# ---------to do: markdown notes---------
#

#
# ---------to-do: optimize code---------
#

# Define the function such that it directly creates the split folder, instead
# of creating an entire folder as a middleman


'''
ENSEMBLES

1. find average of output from both pretrained (parallel)

2. stepwise, put one first then into another (sequential)

'''

Path exists: True
Image shape: Image not loaded


'\nENSEMBLES\n\n1. find average of output from both pretrained (parallel)\n\n2. stepwise, put one first then into another (sequential)\n\n'